In [1]:
import sys
import os

import torch
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import gym
from typing import Any, NamedTuple

from mllib import rllib

In [2]:
envname = 'Taxi-v3'
class ClipRewardEnv(gym.RewardWrapper):
    """
    Clips the reward to {+1, 0, -1} by its sign.
    Args:
        env (gym.Env): The environment to wrap
    """

    def __init__(self, env: gym.Env):
        gym.RewardWrapper.__init__(self, env)
    
    def reward(self, reward: float) -> float:
        return reward/100

def create_env(mode='rgb_array'):
    env = gym.make(envname, render_mode='rgb_array')
    env = rllib.OneHotObservationWrapper(env)
    env = rllib.TorchObservationWrapper(env)
    env = ClipRewardEnv(env)
    return env
env = create_env()

In [3]:
N = 64
class NeuralNetwork(nn.Module):
    def __init__(self, obs_size, act_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_size,N),
            nn.ReLU(),
            nn.Linear(N,N),
            nn.ReLU(),
            nn.Linear(N,act_size),
        )

    def forward(self, x):
        if x.ndim == 1:
            x = torch.reshape(x, (-1,) + x.shape)
        return self.net(x)


def create_model():
    return NeuralNetwork(env.observation_space.shape[0], env.action_space.n)

In [4]:
import torch
from torch.utils.tensorboard import SummaryWriter
from ray import tune
from ray.tune.search import ConcurrencyLimiter

import ray
from ray.tune.search.hyperopt import HyperOptSearch
logdir = '/home/sunho/dev/MLStudy/RL/dqn/runs/'

ri = 0
li = 0
def objective(config):
    N = config['N']
    class NeuralNetwork(nn.Module):
        def __init__(self, obs_size, act_size):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(obs_size,N),
                nn.ReLU(),
                nn.Linear(N,N),
                nn.ReLU(),
                nn.Linear(N,act_size),
            )
    
        def forward(self, x):
            if x.ndim == 1:
                x = torch.reshape(x, (-1,) + x.shape)
            return self.net(x)
    global ri
    global li
    ri = 0
    li = 0
    suffix = "lr={},gamma={},replay_buf_size={},batch_size={},train_count={},update_interval={},eps_decay={},N={}".format(
        config["lr"], config["gamma"], config["replay_buf_size"], config["batch_size"], config["train_count"], 
        config["update_interval"], config["eps_decay"], config['N'])
    # writer = SummaryWriter(logdir+suffix)
    writer = SummaryWriter()
    
    def create_model():
        return NeuralNetwork(env.observation_space.shape[0], env.action_space.n)
    model = create_model()
    #300000
    agent = rllib.DQNAgent(model, create_model(), None, env.action_space, 1.0, 0.0, config["eps_decay"])
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    options = rllib.DQNOptions(optimizer)
    options.replay_buf_size = config["replay_buf_size"]
    options.num_steps = 1024000
    options.double_dqn = True
    options.max_episode_len = 128
    options.gamma = config["gamma"]
    options.update_interval = config["update_interval"]
    options.batch_size = config["batch_size"]
    options.train_count =  config["train_count"]
    options.train_interval = 128
    
    def add_reward(x):
        global ri
        writer.add_scalar("Reward/train", x, ri)
        ri += 1
    
    def add_loss(x):
        global li
        for k,v in x.items():
            writer.add_scalar("Loss/" + k, v, li)
        li += 1
    
    options.report_reward = add_reward
    options.report_train = add_loss
    stats = rllib.dqn_train(create_env, agent, options)
    score = np.mean(stats.reward_history[-128:])
    return {"score": score, "model": model, "agent": agent}

agent = objective({
    'lr': 0.001,
    'gamma': 0.9,
    'replay_buf_size': 256*128,
    'batch_size': 128,
    'N': 64,
    'train_count': 1,
    'update_interval': 1000,
    'eps_decay': 700000,
})["agent"]

/opt/miniconda3/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


In [5]:
agent = rllib.DQNAgent(agent.model, create_model(), None, env.action_space, 0.00, 0.00, 500000)
rllib.run_jupyter(create_env, agent, None, 128)

KeyboardInterrupt: 

In [ ]:
torch.save(agent.model.state_dict(), "taxi3.model")

In [13]:
res.get_best_result()

Result(
  metrics={'score': -0.858254952321907},
  path='/home/sunho/ray_results/objective_2024-05-14_17-30-22/objective_ae69d917_36_N=128,batch_size=128,eps_decay=700000,gamma=0.9000,lr=0.0010,replay_buf_size=16384,train_count=1,update_inte_2024-05-14_17-44-47',
  filesystem='local',
  checkpoint=None
)

In [12]:
context = ray.get_context()
print(context.dashboard_url)

AttributeError: module 'ray' has no attribute 'get_context'

In [ ]:

space = {
    'lr': tune.choice([0.001,0.0001]),
    'gamma': tune.choice([0.8, 0.9, 0.99]),
    'replay_buf_size': tune.choice([64*128, 128*128, 256*128]),
    'batch_size': tune.choice([128]),
    'N': tune.choice([48, 64, 128]),
    'train_count': tune.choice([1, 2]),
    'update_interval': tune.choice([1000, 2000, 3000, 5000, 10000]),
    'eps_decay': tune.choice([300000, 500000, 700000]),
}

algo = HyperOptSearch()
algo = ConcurrencyLimiter(algo, max_concurrent=16)
tuner = tune.Tuner(
    objective,
    tune_config=tune.TuneConfig(
        metric="score",
        mode="max",
        search_alg=algo,
        num_samples=64,
    ),
    param_space=space,
)
res = tuner.fit()